# 🧬 ⚙️ CpGPT Fine-tuning Tutorial ⚙️ 🧬

Welcome to the CpGPT Fine-tuning Tutorial! 👋 

In this notebook, we'll walk you through the process of fine-tuning the CpGPT model for your specific DNA methylation tasks.

## Table of Contents

1. [Setup Environment](#1-setup-environment)
2. [Load CpGPT Dependencies](#2-load-cpgpt-dependencies)
3. [Load Pre-trained Model](#3-load-pre-trained-model)
4. [Prepare and Save Finetuning Data](#4-prepare-and-save-finetuning-data)
5. [Generate DNA LLM Embeddings](#5-generate-dna-llm-embeddings)
6. [Setup Custom Dataloaders](#6-setup-custom-dataloaders)
7. [Fine-tune the Model](#7-fine-tune-the-model)
8. [Visualize Training Progress](#8-visualize-training-progress)
9. [Evaluate Model Performance](#9-evaluate-model-performance)
10. [Generate Predictions](#10-generate-predictions)
11. [Next Steps](#11-next-steps)

## 1. Setup Environment

We'll import the necessary Python packages and set up our environment for fine-tuning CpGPT. We'll be using a mix of standard data science libraries and CpGPT-specific modules. We'll also set some important variables that will be used throughout the notebook. Pay attention to these as you may need to adjust them based on your specific setup and requirements. The bulk of the variables are defined here below. The pre-trained model checkpoint alongside DNA embedding files can also be downloaded if not present.

CpGPT model files and DNA-sequence dependencies are hosted on Hugging Face.
The download cell below downloads any missing files into the standard Hugging Face cache and reuses them on later runs; no command-line download is needed.

### 1.1 Download dependencies and define variables

In [ ]:
from cpgpt import download_cpgpt

resources = download_cpgpt(model="small", species="human")

In [ ]:
# Name to save the model
MODEL_NAME = "CpGPT-AltumAge"

# Random seed for reproducibility
RANDOM_SEED = 42

# Directory paths
CHECKPOINT_DIR = "../logs/finetune"
PROCESSED_DIR = "../data/tutorials/processed/finetune"
ARROW_DF_PATH = "../data/tutorials/raw/toy.arrow"  # Subset of Hannum's data (GSE40279)

# Metadata and Data processing
METADATA_COLS = ["age"]

# Training settings
BATCH_SIZE = 8
MAX_STEPS = 10_000
MAX_INPUT_LENGTH = 10_000
LEARNING_RATE = 1e-4
ONLY_TRAIN_condition_decoder = (
    False  # It is usually better to set it to False to update all of the weights
)

# Device configuration
DEVICE = "cuda"

LLM_DEPENDENCIES_DIR = str(resources.dependencies_path)
DEPENDENCIES_DIR = str(resources.dependencies_path)
MODEL_CHECKPOINT_PATH = str(resources.checkpoint_path)
MODEL_CONFIG_PATH = str(resources.config_path)
MODEL_VOCAB_PATH = str(resources.vocab_path) if resources.vocab_path is not None else None

> **⚠️ Warning**
> 
> Ensure that you have a GPU for finetuning the model as training on a CPU is extremely slow even for small datasets.
> 
> Memory requirements might vary mainly depending on the max input length and the batch size. The standard values should work in most GPUs with at least 12 Gb of memory.
> 
> These parameters are just suggestions and can be changed depending on different user requirements.

### 1.2 Define override dictionary

Below, you can see which parameters can be changed to override the defaults when later loading the model.


---
<details>
<summary><strong>net</strong></summary>

The net dictionary contains parameters to configure architecture of the model.

- **d_embedding**
  - Type: int
  - Description: Dimensionality of the embeddings used within the model.
  - Recommendation: Do not change.

- **d_hidden**
  - Type: int
  - Description: Dimensionality of the hidden layers in the MLP blocks.
  - Recommendation: Do not change.

- **d_dna_embedding**
  - Type: int
  - Description: Dimensionality of the DNA embeddings input to the model.
  - Recommendation: Only change if you would like to try the embeddings of a DNA LLM not used in the pre-training.

- **n_attention_heads**
  - Type: int
  - Description: Number of attention heads in the transformer encoder layers.
  - Recommendation: Do not change.

- **n_layers**
  - Type: int
  - Description: Number of layers in the transformer encoder.
  - Recommendation: Do not change.

- **n_mlp_blocks**
  - Type: int
  - Description: Number of MLP blocks in the decoder modules.
  - Recommendation: Do not change.

- **dropout**
  - Type: float
  - Description: Dropout rate applied in various parts of the model to prevent overfitting.
  - Recommendation: Only change if the model seems to be overfitting, which is highly unlikely.

- **positional_encoding**
  - Type: str
  - Description: Type of positional encoding to apply. Options are:
    - 'none': No positional encoding.
    - 'rotary': Apply rotary positional encoding.
    - 'chrom': Use chromosome positional encoding.
    - 'hic': Use Hi-C interaction-based positional encoding.
  - Recommendation: Change to 'none' if GPU memory is a bottleneck as 'hic' is memory-intensive.

- **sample_embedding_method**
  - Type: str
  - Description: Method to obtain the sample-level embedding from site-level embeddings. Options are:
    - 'cls': Use the CLS token embedding.
    - 'mean': Use the mean of all site embeddings.
    - 'max': Use the maximum across all site embeddings.
  - Recommendation: Some studies show better performance for downstream tasks when using 'mean' rather than 'cls', but keep in mind that the model was pre-trained with 'cls'.

- **use_power_norm**
  - Type: bool
  - Description: Whether to apply power normalization to the sample embeddings.
  - Recommendation: It helps faster convergence but might slightly hurt the performance.

- **fft**
  - Type: bool
  - Description: Whether to use Fast Fourier Transform-based layers.
  - Recommendation: Only change if you do not want to use attention in the transformer, which can allow very long sequences (100k CpG sites). However, given the same input size, FFT layers perform wose than attention.

- **use_condition_decoder**
  - Type: bool
  - Description: Whether to include a query decoder in the model for predicting additional targets (e.g., age).
  - Recommendation: Change it to True for predicting a condition.

- **condition_size**
  - Type: int
  - Description: Number of query conditions (targets) to predict if use_condition_decoder is True.
  - Recommendation: This is the shape of the conditions.

</details>

---
<details>
<summary><strong>training</strong></summary>

The training dictionary contains parameters related to the training process.

- **style**
  - Type: str
  - Description: Training style or strategy to use for the model. Options are:
    - 'bert': Masked language modeling approach.
    - 'gpt': Autoregressive training.
  - Recommendation: Change to 'bert' for finetuning.

- **binarize_input**
  - Type: bool
  - Description: Whether to binarize the input methylation values (e.g., convert to 0 or 1).
  - Recommendation: Only binarize the input if working with single-cell methylation data.

- **contrastive_threshold**
  - Type: float
  - Description: Threshold for contrastive loss when distinguishing between similar and dissimilar samples.
  - Recommendation: Do not change.

- **loss_weights**
  - Type: dict
  - Description: Weights for different loss components used during training. Keys and their descriptions:
    - **betas_mae**
      - Type: float
      - Description: Weight for the Mean Absolute Error loss between predicted and true beta values.
    - **betas_mae_unc**
      - Type: float
      - Description: Weight for the uncertainty MAE loss.
    - **betas_kld**
      - Type: float
      - Description: Weight for the Kullback-Leibler divergence loss on beta values.
    - **betas_beta**
      - Type: float
      - Description: Weight for the Beta distribution loss on methylation values.
    - **betas_wd**
      - Type: float
      - Description: Weight for the Wasserstein distance loss on betas.
    - **contrastive**
      - Type: float
      - Description: Weight for the contrastive loss between sample embeddings.
    - **sample_kld**
      - Type: float
      - Description: Weight for the KLD loss on sample embeddings.
    - **condition_loss**
      - Type: float
      - Description: Weight for the loss associated with the query decoder.

- **condition_decoder_loss**
  - Type: str
  - Description: Specifies the loss function to use for the query decoder. Options include:
    - 'mae': Mean Absolute Error loss.
    - 'mse': Mean Squared Error loss.
    - 'ce': Cross-Entropy loss. The model predicts the logits which need to be transformed back to probabilities.
  - Recommendation: Change depending on your condition.
</details>

---

<details>
<summary><strong>compile</strong></summary>

- **Type**: bool
- **Description**: Determines whether to compile the model using PyTorch's JIT compilation for potential speed improvements. Set to True to compile the model.
- Recommendation: Change to False as compiling might make the model more difficult to debug.
</details>

---

In [ ]:
# Define override dictionary
override_dict = {
    "net": {
        "use_condition_decoder": True,
        "condition_size": 1,
    },
    "training": {
        "style": "bert",
        "condition_decoder_loss": "mae",
        "loss_weights": {
            "condition_loss": 1,
        },
    },
    "compile": False,
}

### 1.3 Import packages


In [ ]:
# Standard library imports
import os
import warnings

warnings.simplefilter(action="ignore", category=FutureWarning)

import numpy as np
import pandas as pd

# Torch imports
# Lightning imports
from lightning.pytorch import Trainer, seed_everything
from lightning.pytorch.callbacks import (
    EarlyStopping,
    LearningRateMonitor,
    ModelCheckpoint,
    RichModelSummary,
    RichProgressBar,
)
from lightning.pytorch.loggers import TensorBoardLogger

# Sklearn imports
from sklearn.model_selection import train_test_split
from torch.optim import AdamW
from torch.utils.data import DataLoader

# TorchTune imports
from torchtune.modules import get_cosine_schedule_with_warmup

# cpgpt-specific imports
from cpgpt.data.components.cpgpt_datasaver import CpGPTDataSaver
from cpgpt.data.components.cpgpt_dataset import CpGPTDataset, cpgpt_data_collate
from cpgpt.data.components.dna_llm_embedder import DNALLMEmbedder
from cpgpt.data.components.illumina_methylation_prober import IlluminaMethylationProber
from cpgpt.infer.cpgpt_inferencer import CpGPTInferencer
from cpgpt.model.utils import SaveDataModuleHyperparametersCallback
from cpgpt.utils.rich_utils import create_rich_dataset_preview, print_rich_model_info

# Set random seed for reproducibility
seed_everything(RANDOM_SEED, workers=True)

## 2. Load CpGPT Dependencies

Now that we have our basic environment set up, we need to load three crucial components for CpGPT:

1. **DNA LLM Embedder**: This component contains the embeddings for all CpGs of interest. It's essential for converting DNA sequences into a format that our model can understand and process.

2. **Illumina Methylation Prober**: This tool allows us to convert probe names to genomic locations, which is required for working with Illumina methylation array data.

3. **CpG Inferencer**: This component is responsible for loading the pre-trained CpGPT model checkpoint and any additional configurations or overrides needed for the fine-tuning process.

### 3.1 Load DNALLMEmbedder

In [ ]:
embedder = DNALLMEmbedder(dependencies_dir=DEPENDENCIES_DIR)

### 3.2 Load IlluminaMethylationProber

In [ ]:
prober = IlluminaMethylationProber(dependencies_dir=DEPENDENCIES_DIR, embedder=embedder)

### 3.3 Load CpGInferencer

In [ ]:
inferencer = CpGPTInferencer()

## 3. Load Pre-trained Model

We need to load a pre-trained CpGPT model that we'll fine-tune on our specific task. Loading a pre-trained model allows us to leverage the general DNA methylation knowledge that CpGPT has already learned, potentially improving our fine-tuning results and reducing training time. We can override any setting from the pre-trained model by using a dictionary with the new arguments and variables.

### 3.1 Load checkpoint

In [ ]:
from omegaconf import OmegaConf

original_config = inferencer.load_cpgpt_config(MODEL_CONFIG_PATH)
config = OmegaConf.merge(original_config, override_dict)
model_info = {
    "original_hparams": original_config,
    "current_hparams": config,
}
model = inferencer.load_cpgpt_model(
    config,
    model_ckpt_path=MODEL_CHECKPOINT_PATH,
    strict_load=True,
)

### 3.2 Print model info

In [ ]:
# Check overriden parameters
print_rich_model_info(model_info, print_changes_only=True)

## 4. Prepare and Save Finetuning Data

We'll prepare our methylation data for fine-tuning and save it in a format that CpGPT can efficiently process. This step is necessary for ensuring that our fine-tuning process runs smoothly and that we have proper validation and test sets for evaluating our model's performance. 

We first begin with a dataset that is already stored in .arrow (feather v2) format. It is recommended to pre-filter CpG sites, as the more there are, the longer and the harder it will be for the model to link the methylome to the variable. Generally, 5000 to 10000 CpG sites works well. Depending on the difficulty of the task (and your patience for training), feature sizes above 20000 CpG sites might be tricky.

There is no need to impute missing methylation values as CpGPT will simply ignore them. In fact, imputing missing values beforehand might worsen the results as imputation methods (besides CpGPT's of course) are imperfect.

### 4.1 Create train, val, test splits

In [ ]:
# Load your DataFrame
df = pd.read_feather(ARROW_DF_PATH)

# First, split off the test set (20% of the total data)
train_val, test = train_test_split(df, test_size=0.2, random_state=RANDOM_SEED)

# Then, split the remaining data into training and validation sets
train, val = train_test_split(train_val, test_size=0.05, random_state=RANDOM_SEED)

# Directories for saving the splits
train_dir = os.path.join(PROCESSED_DIR, "train")
val_dir = os.path.join(PROCESSED_DIR, "val")
test_dir = os.path.join(PROCESSED_DIR, "test")

# Create directories for saving the splits
Path(train_dir).mkdir(parents=True, exist_ok=True)
Path(val_dir).mkdir(parents=True, exist_ok=True)
Path(test_dir).mkdir(parents=True, exist_ok=True)

# Paths for saving the splits
train_path = os.path.join(train_dir, "train.arrow")
val_path = os.path.join(val_dir, "val.arrow")
test_path = os.path.join(test_dir, "test.arrow")

# Save the splits
train.to_feather(train_path)
val.to_feather(val_path)
test.to_feather(test_path)

### 4.2 Memory-map splits

In [ ]:
# Define datasavers
train_datasaver = CpGPTDataSaver(
    data_paths=train_path,
    metadata_cols=METADATA_COLS,
    processed_dir=train_dir,
)
val_datasaver = CpGPTDataSaver(
    data_paths=val_path,
    metadata_cols=METADATA_COLS,
    processed_dir=val_dir,
)
test_datasaver = CpGPTDataSaver(
    data_paths=test_path,
    metadata_cols=METADATA_COLS,
    processed_dir=test_dir,
)

# Process the files
train_datasaver.process_files(prober, embedder)
val_datasaver.process_files(prober, embedder)
test_datasaver.process_files(prober, embedder)

## 5. Generate DNA LLM Embeddings
 
In this section, we'll generate any DNA embeddings for the genomic locations of our datasets that might be missing. The model requires that all genomic locations have embeddings from a DNA LLM or it will fail otherwise. The easiest way is to download such embeddings if you are working with human samples ([step 1.4](#1.4-download-dependencies)) and Illumina arrays or it might take a little bit for the processing to happen.

In [ ]:
# Get all unique species from the datasavers
all_species = (
    set(train_datasaver.all_genomic_locations.keys())
    | set(val_datasaver.all_genomic_locations.keys())
    | set(test_datasaver.all_genomic_locations.keys())
)

# Loop through each species
for species in all_species:
    all_genomic_locations = np.unique(
        list(train_datasaver.all_genomic_locations.get(species, []))
        + list(val_datasaver.all_genomic_locations.get(species, []))
        + list(test_datasaver.all_genomic_locations.get(species, [])),
    )

    embedder.parse_dna_embeddings(
        all_genomic_locations,
        species,
        dna_llm=model_info["current_hparams"]["data"]["dna_llm"],
        dna_context_len=model_info["current_hparams"]["data"]["dna_context_len"],
    )

## 6. Setup Custom Dataloaders

We'll set up custom dataloaders for our training, validation, and test sets. This process involves creating CpGPTDataset objects for each dataset split, configuring dataloader parameters for optimal performance, and verifying the output of our dataloaders.

### 6.1 Create CpGPTDataset objects

In [ ]:
# Train
train_data = CpGPTDataset(
    embedder,
    processed_dir=train_dir,
    max_length=MAX_INPUT_LENGTH,
    sorting_strategy=model_info["current_hparams"]["data"]["sorting_strategy"],
    dna_context_len=model_info["current_hparams"]["data"]["dna_context_len"],
    dna_llm=model_info["current_hparams"]["data"]["dna_llm"],
)

# Validation
val_data = CpGPTDataset(
    embedder,
    processed_dir=val_dir,
    max_length=MAX_INPUT_LENGTH,
    sorting_strategy=model_info["current_hparams"]["data"]["sorting_strategy"],
    dna_context_len=model_info["current_hparams"]["data"]["dna_context_len"],
    dna_llm=model_info["current_hparams"]["data"]["dna_llm"],
)

# Test
test_data = CpGPTDataset(
    embedder,
    processed_dir=test_dir,
    max_length=MAX_INPUT_LENGTH,
    sorting_strategy=model_info["current_hparams"]["data"]["sorting_strategy"],
    dna_context_len=model_info["current_hparams"]["data"]["dna_context_len"],
    dna_llm=model_info["current_hparams"]["data"]["dna_llm"],
)

Let's double check the outputs make sense:

In [ ]:
# Train Data Sample
create_rich_dataset_preview(train_data, "Train Data Sample")

# Validation Data Sample
create_rich_dataset_preview(val_data, "Validation Data Sample")

# Test Data Sample
create_rich_dataset_preview(test_data, "Test Data Sample")

### 6.2 Create Dataloaders

Now that we have created our datasets, we'll set up the actual DataLoader objects. These will be responsible for batching our data and shuffling it during training. Let's set them up:

In [ ]:
# Train
train_dataloader = DataLoader(
    dataset=train_data,
    batch_size=BATCH_SIZE,
    num_workers=4,
    shuffle=True,
    collate_fn=cpgpt_data_collate,
    drop_last=False,
)

# Validation
val_dataloader = DataLoader(
    dataset=val_data,
    batch_size=BATCH_SIZE,
    num_workers=4,
    shuffle=False,
    collate_fn=cpgpt_data_collate,
    drop_last=False,
)

# Test
test_dataloader = DataLoader(
    dataset=test_data,
    batch_size=4,
    num_workers=0,
    shuffle=False,
    collate_fn=cpgpt_data_collate,
    drop_last=False,
)

## 7. Fine-tune the Model

With our data prepared and model loaded, we're now ready to set up the fine-tuning process. This involves configuring several components:

1. Callbacks for model checkpointing and early stopping
2. A logger for tracking training progress
3. The Trainer object, which orchestrates the training process
4. Optimizer and learning rate scheduler

### 7.1 Define Trainer parameters

In [ ]:
# Define all callbacks and loggers
datamodule_hyper_parameters = {
    **model_info["current_hparams"]["data"],
}
save_datamodule_hyperparameters_callback = SaveDataModuleHyperparametersCallback(
    datamodule_hyper_parameters,
)

checkpoint_callback = ModelCheckpoint(
    dirpath=CHECKPOINT_DIR,
    filename=f"{MODEL_NAME}/{{epoch:02d}}-{{step:02d}}",
    monitor="val/condition_loss",
    verbose=False,
    save_last=True,
    save_top_k=1,
    mode="min",
)

early_stopping_callback = EarlyStopping(
    monitor="val/loss",
    patience=100,
    min_delta=0.01,
    check_finite=True,
)

model_summary_callback = RichModelSummary()

lr_monitor = LearningRateMonitor(logging_interval="step")

rich_progress_bar = RichProgressBar()

logger = TensorBoardLogger(
    save_dir=CHECKPOINT_DIR,
)  # Alternatively, use CSVLogger(save_dir=CHECKPOINT_DIR) if TensorBoard not available

In [ ]:
# Create Trainer
trainer = Trainer(
    min_steps=MAX_STEPS // 10,
    max_steps=MAX_STEPS,
    enable_progress_bar=True,
    enable_model_summary=True,
    devices="auto",
    accelerator="auto",
    precision="16-mixed",
    log_every_n_steps=10,
    val_check_interval=100,  # If there is a lot of training data, we can check the validation less frequently
    check_val_every_n_epoch=None,
    callbacks=[
        checkpoint_callback,
        early_stopping_callback,
        model_summary_callback,
        rich_progress_bar,
        lr_monitor,
        save_datamodule_hyperparameters_callback,
    ],
    logger=logger,
)

### 7.2 Reset optimizer and scheduler

In [ ]:
# Conditional freezing based on ONLY_TRAIN_condition_decoder
if ONLY_TRAIN_condition_decoder:
    # Freeze all parameters
    for param in model.parameters():
        param.requires_grad = False

    # Unfreeze parameters related to condition tokens
    condition_params = []
    for name, param in model.named_parameters():
        if "condition_tokens" in name or "condition_decoder" in name:
            param.requires_grad = True
            condition_params.append(param)
    params = condition_params
else:
    # Train all parameters
    params = model.parameters()

# Define optimizer and scheduler
optimizer = AdamW(params, lr=LEARNING_RATE, weight_decay=0.1, betas=(0.9, 0.95))
scheduler = get_cosine_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=MAX_STEPS // 10,
    num_training_steps=MAX_STEPS,
)

In [ ]:
# Reset model optimizer and scheduler
model.configure_optimizers = lambda: {
    "optimizer": optimizer,
    "lr_scheduler": {
        "scheduler": scheduler,
        "interval": "step",
        "frequency": 1,
    },
}

### 7.3 Fit the model

In [ ]:
trainer.fit(model, train_dataloader, val_dataloader)

## 8. Visualize Training Progress

Monitoring the training process is crucial for understanding how our model is learning and identifying any potential issues. We'll use TensorBoard to visualize our training progress. We'll visualize:

1. Training and validation loss over time
2. Learning rate changes throughout training
3. Other relevant metrics specific to our task

These visualizations will help us ensure that our model is learning effectively and allow us to make informed decisions about potential hyperparameter adjustments.

In [ ]:
%reload_ext tensorboard
%tensorboard --logdir={CHECKPOINT_DIR}/{MODEL_NAME}

## 9. Evaluate Model Performance

Now that we have fine-tuned our model, it's crucial to evaluate its performance rigorously. This evaluation will help us understand how well our model generalizes to unseen data and how it compares to baseline methods or previous versions.

In [ ]:
test_results = trainer.test(dataloaders=test_dataloader)

## 10. Generate Predictions

With our model fine-tuned and evaluated, we can now use it to generate predictions on our datasets. This will be shown in more detail in another notebook geared towards model inference and evaluation. However, a simple example is provided below on how to generate predictions for the test dataset.

### 10.1 Run inference

In [ ]:
# Let's embed our samples
sample_embedding = inferencer.embed_sample(model, data=test_data)

In [ ]:
# From the embeddings, let's predict the condition values
predictions = inferencer.query_condition(model, sample_embedding=sample_embedding)

In [ ]:
# Show first few predictions
predictions[0:5]

## 11. Next steps

Here are some ideas for further exploration:

- Experiment with different hyperparameters to optimize performance
- Conduct more in-depth analysis of your model's predictions
- Explore transfer learning by fine-tuning on related tasks